# Semana 13 - EC2 e Kubernetes na AWS com a aplicação Copa
## Aula prática: duas imagens públicas do Docker Hub

Nesta aula vamos baixar e implantar as imagens públicas `souzaandreprof/copa-frontend:latest` e `souzaandreprof/copa-backend:latest`. O frontend usa Nginx na porta 80; o backend Node.js escuta na porta 3001.

A arquitetura prática será: frontend acessivel pela web em TCP/80 e backend dentro da rede da aplicação em TCP/3001. Em produção, HTTPS/443 pode terminar em um Application Load Balancer com dominio e certificado AWS Certificate Manager.

**Observação sobre estas imagens:** o frontend publicado tem o endereço da API compilado no JavaScript. Antes da aula, confira se ele aponta para o backend desejado. O backend tem `/health` na porta 3001. O teste de rede interno abaixo valida o backend mesmo se a interface publicada estiver apontando para outro endereço.

## Objetivos

- localizar VPC, sub-redes, rotas, Security Groups e EC2 no Console AWS;
- baixar duas imagens públicas do Docker Hub;
- executar o frontend na porta 80 e manter o backend na porta 3001;
- comparar EC2 (IaaS) com Kubernetes gerenciado (EKS);
- testar conectividade permitida e encerrar recursos de laboratorio.

## Mapa da prática

```text
Navegador --TCP/80--> frontend Copa (Nginx)
                           |
                           +-- teste interno TCP/3001 --> backend Copa (Node.js)

EC2: instancias e Security Groups
EKS: Deployment + Pods + Service LoadBalancer (frontend) + ClusterIP (backend)
```

A imagem frontend atualmente possui uma URL de API compilada. Se o endpoint não for o mesmo backend implantado pela turma, a página não vai consumir esse backend sem uma nova build de frontend. O objetivo de rede da aula continua sendo verificar que o Service privado do backend responde apenas a partir da rede autorizada.

## 1. Preparação e custo

1. Inicie o AWS Academy Learner Lab e abra o Console.
2. Confirme a região da turma e use a mesma região em todos os recursos.
3. Verifique com o professor se EKS, CloudShell e as permissões IAM estão habilitados.
4. Anote o saldo/credito. EKS, nós e Load Balancers podem gerar custos enquanto ativos.

Se o Learner Lab bloquear EKS, conclua a prática de EC2 e faca a comparação guiada. Não tente contornar permissões do laboratorio.

## 2. Reconhecer a infraestrutura no Console

Em VPC > Your VPCs, Subnets e Route tables, identifique a VPC, as sub-redes e a rota pública para o Internet Gateway. Em EC2 > Instances, observe a instância da aula e anote:

| Campo | O que observar |
|---|---|
| Instance state / type | estado e tamanho da VM |
| AMI | sistema operacional |
| VPC / Subnet | rede e segmento |
| Private IPv4 / Public IPv4 | endereços e exposição |
| Security groups | portas e origens permitidas |
| Storage | volume EBS |

Na arquitetura em duas camadas, o frontend fica em subnet pública com TCP/80. O backend fica em subnet privada, sem IP publico, e recebe TCP/3001 somente do Security Group do frontend. Para HTTPS/443, use ALB e certificado ACM como extensao.

## 3. IaaS: baixar e executar as imagens nas EC2

Use duas instâncias pequenas permitidas pelo Learner Lab: uma pública para o frontend e outra privada para o backend. Configure os Security Groups conforme a tabela:

| Security Group | Regra de entrada |
|---|---|
| `copa-sg-front` | TCP/80 da internet; administração restrita ao IP do aluno/método do lab |
| `copa-sg-back` | TCP/3001 com origem `copa-sg-front`; sem IP publico e sem SSH publico |

Instale Docker em cada instância Amazon Linux:

```bash
sudo dnf install -y docker
sudo systemctl enable --now docker
```

Na EC2 privada do backend, baixe e inicie a imagem:

```bash
sudo docker pull souzaandreprof/copa-backend:latest
sudo docker run -d --name copa-backend --restart unless-stopped -p 3001:3001 souzaandreprof/copa-backend:latest
sudo docker ps
```

Na EC2 pública do frontend:

```bash
sudo docker pull souzaandreprof/copa-frontend:latest
sudo docker run -d --name copa-frontend --restart unless-stopped -p 80:80 souzaandreprof/copa-frontend:latest
sudo docker ps
```

Acesse `http://IP_PUBLICO_FRONTEND`. Para testar a rota privada, conecte-se a EC2 frontend e execute `curl http://IP_PRIVADO_BACKEND:3001/health`. A resposta esperada e JSON com `status: ok`.

**Verificação importante:** o bundle JavaScript da imagem frontend publicada contém uma URL de API fixa. Confira no navegador (Developer Tools > Network) se as requisicoes vao para o backend implantado. Se o endereço fixo não corresponder ao backend da aula, o frontend precisara ser atualizado e publicado para integrar a interface; não exponha o backend a internet para contornar isso.

## 4. Comparação: EC2 (IaaS) e EKS (Kubernetes gerenciado)

| Aspecto | EC2 com Docker | Amazon EKS |
|---|---|---|
| Host | equipe administra as VMs | AWS administra o plano de controle; nós são configurados a parte |
| Execução | Docker inicia cada container | Deployment agenda e mantem Pods |
| Rede externa | IP, subnet e Security Group | Service LoadBalancer |
| Rede interna | IP privado e Security Group de origem | Service ClusterIP e DNS do cluster |
| Imagens | Docker Hub | Docker Hub |

EKS e um serviço Kubernetes gerenciado (CaaS, frequentemente ensinado junto a PaaS). A equipe ainda administra aplicação, identidade, rede, capacidade e custos.

## 5. Implantar no Amazon EKS (se habilitado)

### 5.1 Criar cluster e capacidade

No Console, abra Amazon EKS > Clusters > Create cluster e crie `aula-k8s` com as roles, VPCs e sub-redes autorizadas pelo Learner Lab. Adicione capacidade computacional pelo fluxo indicado no Console e aguarde os nós ficarem `Ready`.

Se a criação de role, cluster ou nós for negada, registre a mensagem e siga a alternativa guiada. Não altere políticas IAM para contornar o bloqueio.

### 5.2 Deploy das duas imagens e Services

No CloudShell da mesma região, configure o `kubectl` seguindo EKS > cluster > Connect e confira `kubectl get nodes`.

Crie os Deployments com as imagens públicas:

```bash
kubectl create deployment copa-frontend --image=souzaandreprof/copa-frontend:latest --replicas=2
kubectl create deployment copa-backend --image=souzaandreprof/copa-backend:latest --replicas=1
kubectl rollout status deployment/copa-frontend
kubectl rollout status deployment/copa-backend
kubectl get deployments,pods -o wide
```

Exponha somente o frontend por Load Balancer. Mantenha o backend privado dentro do cluster:

```bash
kubectl expose deployment copa-frontend --type=LoadBalancer --port=80 --target-port=80
kubectl expose deployment copa-backend --type=ClusterIP --port=3001 --target-port=3001
kubectl get services --watch
```

Quando o endereço externo do frontend aparecer, abra `http://ENDERECO_DO_SERVICE`. Teste a rede interna a partir de um Pod frontend:

```bash
kubectl exec deployment/copa-frontend -- wget -qO- http://copa-backend:3001/health
```

A resposta esperada e `status: ok`. O Service do backend não recebe endereço publico. O Load Balancer pode gerar custos; remova-o ao fim da prática.

## 6. Observar e explicar

```bash
kubectl get nodes
kubectl get deployments,pods -o wide
kubectl get services
kubectl describe service copa-backend
kubectl logs deployment/copa-backend
```

Responda:

1. Qual imagem e tag executa em cada Deployment?
2. Qual Service tem endereço externo e qual tem somente DNS interno?
3. Por que o backend não precisa de IP publico para receber chamadas dos Pods na mesma rede?
4. O que acontece com os Pods quando um Pod e removido?
5. Que recursos AWS geram custo nesta arquitetura?

Remova um Pod frontend e observe o Deployment criar outro:

```bash
kubectl delete pod NOME_DE_UM_POD_FRONTEND
kubectl get pods --watch
```

Interrompa `--watch` com Ctrl+C depois que o novo Pod aparecer.

## 7. Diagnóstico

| Sintoma | Verificação inicial |
|---|---|
| frontend não abre | IP/Load Balancer, rota pública, TCP/80 e estado do Nginx |
| `/health` não responde | IP privado/Service DNS, TCP/3001, Security Group ou Pod pronto |
| Pod `ImagePullBackOff` | nome/tag pública e saída de rede dos nós |
| Service sem endereço externo | tipo do Service, sub-redes, permissao e tempo do Load Balancer |
| interface não mostra dados da API | URL compilada no frontend, CORS e conectividade do endpoint configurado |

Mude uma condicao por vez. Não abra todas as portas para corrigir um timeout.

## 8. Alternativa quando EKS não está disponível

1. Conclua a prática com as duas EC2 e imagens Docker Hub.
2. No Console, examine Amazon EKS e identifique cluster, VPC/sub-redes, nós e roles IAM sem criar recursos.
3. Desenhe frontend LoadBalancer e backend ClusterIP.
4. Registre qualquer limitacao do Learner Lab; não tente contornar permissões.

## 9. Limpeza obrigatória

No EKS, remova primeiro os Services e Deployments:

```bash
kubectl delete service copa-frontend copa-backend
kubectl delete deployment copa-frontend copa-backend
```

Depois exclua capacidade computacional e cluster `aula-k8s`. Confirme que o Load Balancer foi removido.

Na EC2, remova os containers (`sudo docker rm -f copa-frontend` e `sudo docker rm -f copa-backend`) e encerre as instâncias descartaveis. Libere apenas Elastic IPs e recursos criados para a aula. Encerre com End Lab.

## Entrega da prática

Entregue uma página com:

- diagrama frontend publico / backend privado;
- evidência dos `docker pull` e do frontend em TCP/80;
- resposta de `/health` em TCP/3001 por caminho interno;
- capturas do Console para EC2/Security Groups ou EKS/Pods/Services;
- comparação IaaS e Kubernetes gerenciado;
- registro da URL de API compilada no frontend e da limpeza dos recursos.

## Referências oficiais

- Imagem frontend no Docker Hub: <https://hub.docker.com/repository/docker/souzaandreprof/copa-frontend>
- Imagem backend no Docker Hub: <https://hub.docker.com/repository/docker/souzaandreprof/copa-backend>
- AWS Amazon EKS: <https://docs.aws.amazon.com/eks/latest/userguide/what-is-eks.html>
- Kubernetes Deployments: <https://kubernetes.io/docs/concepts/workloads/controllers/deployment/>
- Kubernetes Services: <https://kubernetes.io/docs/concepts/services-networking/service/>